# Phase 3: Logit-Space Kalman Filter

The raw Kalman filter can produce estimates outside [0,1], which is invalid for probabilities.
The logit-space filter transforms observations to log-odds space where the domain is unbounded,
runs the filter there, and transforms back. This guarantees bounded output and produces
asymmetric confidence intervals near the boundaries.

## Key demonstrations
1. Regular filter exceeding bounds vs logit filter staying bounded
2. Asymmetric confidence intervals near p=0.9+
3. Comparison across markets at different price ranges

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from loguru import logger
logger.disable('src')

from src.data.synthetic import generate_random_walk
from src.filters.scalar_kalman import ScalarKalmanFilter
from src.filters.logit_kalman import LogitKalmanFilter
from src.utils.transforms import logit, sigmoid

sns.set_theme(style='whitegrid')
%matplotlib inline

## 1. Bounded vs Unbounded Filter Near p=0.95

In [ ]:
# Generate data near 0.95 (approaching resolution)
data = generate_random_walk(n_steps=300, Q=5e-4, R=5e-3, x0=0.92, seed=42)

# Regular Kalman filter
regular = ScalarKalmanFilter(Q=5e-4, R=5e-3)
result_reg = regular.filter(data.observations)

# Logit Kalman filter
logit_f = LogitKalmanFilter(Q_logit=5e-3, R_prob=5e-3)
result_logit = logit_f.filter(data.observations)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

# Regular filter
ax1.plot(data.observations, color='gray', alpha=0.4, linewidth=0.8, label='Raw')
ax1.plot(result_reg.states, color='#00CED1', linewidth=1.5, label='Regular Kalman')
std_reg = np.sqrt(result_reg.covariances) * 2
ax1.fill_between(range(300), result_reg.states - std_reg, result_reg.states + std_reg,
                 color='#00CED1', alpha=0.15, label='\u00b12\u03c3 CI')
ax1.axhline(y=1.0, color='red', linestyle='--', alpha=0.5, label='y=1.0 boundary')
ax1.axhline(y=0.0, color='red', linestyle='--', alpha=0.5)
exceeded = np.sum((result_reg.states + std_reg) > 1.0)
ax1.set_title(f'Regular Filter: CI exceeds 1.0 at {exceeded} points')
ax1.legend()
ax1.set_ylabel('Probability')

# Logit filter
ax2.plot(data.observations, color='gray', alpha=0.4, linewidth=0.8, label='Raw')
ax2.plot(result_logit.states_prob, color='#FF6347', linewidth=1.5, label='Logit Kalman')
ax2.fill_between(range(300), result_logit.lower_95, result_logit.upper_95,
                 color='#FF6347', alpha=0.15, label='\u00b12\u03c3 CI (bounded)')
ax2.axhline(y=1.0, color='red', linestyle='--', alpha=0.5)
ax2.axhline(y=0.0, color='red', linestyle='--', alpha=0.5)
ax2.set_title('Logit Filter: CI always in [0, 1]')
ax2.legend()
ax2.set_xlabel('Time step')
ax2.set_ylabel('Probability')

plt.tight_layout()
plt.show()

## 2. Asymmetric Confidence Intervals

In [ ]:
# Show CI asymmetry at different probability levels
probs = [0.1, 0.3, 0.5, 0.7, 0.9]

fig, axes = plt.subplots(1, len(probs), figsize=(16, 4))

for ax, p0 in zip(axes, probs):
    data = generate_random_walk(n_steps=150, Q=1e-4, R=2e-3, x0=p0, seed=42)
    lkf = LogitKalmanFilter(Q_logit=1e-3, R_prob=2e-3)
    result = lkf.filter(data.observations)

    ax.plot(result.states_prob, color='#FF6347', linewidth=1.5)
    ax.fill_between(range(150), result.lower_95, result.upper_95,
                    color='#FF6347', alpha=0.15)
    ax.set_title(f'p \u2248 {p0}')
    ax.set_ylim(-0.05, 1.05)
    ax.axhline(y=0, color='gray', linewidth=0.5)
    ax.axhline(y=1, color='gray', linewidth=0.5)

axes[0].set_ylabel('Probability')
fig.suptitle('Logit Filter: Asymmetric CIs at Different Price Levels', y=1.02)
plt.tight_layout()
plt.show()

## 3. Logit Space Visualization

In [ ]:
data = generate_random_walk(n_steps=300, Q=3e-4, R=3e-3, x0=0.7, seed=42)
lkf = LogitKalmanFilter(Q_logit=5e-3, R_prob=3e-3)
result = lkf.filter(data.observations)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8))

# Probability space
ax1.plot(data.observations, color='gray', alpha=0.4, linewidth=0.8, label='Raw')
ax1.plot(result.states_prob, color='#FF6347', linewidth=1.5, label='Filtered')
ax1.fill_between(range(300), result.lower_95, result.upper_95,
                 color='#FF6347', alpha=0.15, label='95% CI')
ax1.set_title('Probability Space')
ax1.legend()
ax1.set_ylabel('Probability')

# Logit space
obs_logit = np.array([float(logit(np.clip(z, 1e-10, 1-1e-10))) for z in data.observations])
ax2.plot(obs_logit, color='gray', alpha=0.4, linewidth=0.8, label='Raw (logit)')
ax2.plot(result.states_logit, color='#FF6347', linewidth=1.5, label='Filtered (logit)')
std_logit = np.sqrt(result.covariances_logit) * 2
ax2.fill_between(range(300), result.states_logit - std_logit, result.states_logit + std_logit,
                 color='#FF6347', alpha=0.15, label='\u00b12\u03c3 (logit)')
ax2.set_title('Logit Space (symmetric CIs here)')
ax2.legend()
ax2.set_xlabel('Time step')
ax2.set_ylabel('Logit(p)')

plt.tight_layout()
plt.show()